The goal of this activity is to demonstrate how data size can affect the accuracy of deep learning models. In the cell below, please enter a number between 64 (the batch size) and 60,000 (max data sample)) that will determine how many samples the data set is trained on. Feel free to go to larger epochs or change the batch size as well.

In [4]:
sample_size = 60000 #TODO - change this number
set_epochs = 10
batch = 64

The cell below is a concatination of the first few cells of the Week 10 (Japanese character recognition) lab, updated to use your above parameter to adjust.

In [5]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

import tensorflow.keras as keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator

with np.load("kmnist-train-imgs.npz") as training_images_fl:
    training_images = training_images_fl["arr_0"]
    
with np.load("kmnist-test-imgs.npz") as testing_images_fl:
    testing_images = testing_images_fl["arr_0"]
    
with np.load("kmnist-train-labels.npz") as training_labels_fl:
    training_labels = training_labels_fl["arr_0"]
    
with np.load("kmnist-test-labels.npz") as testing_labels_fl:
    testing_labels = testing_labels_fl["arr_0"]
    
    
num_classes = 10

# input image dimensions
img_rows, img_cols = 28, 28

x_train = training_images
x_test = testing_images
# Convert class vectors to binary class matrices
y_train = training_labels
y_test = testing_labels

if K.image_data_format() == 'channels_first':
    x_train = x_train.reshape(x_train.shape[0], 1, img_rows, img_cols)
    x_test = x_test.reshape(x_test.shape[0], 1, img_rows, img_cols)
    input_shape = (1, img_rows, img_cols)
else:
    x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
    x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)
    input_shape = (img_rows, img_cols, 1)

x_train = x_train/255
x_test = x_test/255

#resize to smaller training data
x_train = x_train[:sample_size]
y_train = y_train[:sample_size]

train_samples = len(x_train)
print(x_train.shape)
test_samples = len(x_test)

# create the data generator to feed the training with augmentations
datagen = ImageDataGenerator(rotation_range=1,zoom_range=0.01)
datagen.fit(x_train)

# Create the model
dnn_epochs = set_epochs
dnn_batch_size = batch
steps_per_epoch = (train_samples + dnn_batch_size - 1) // dnn_batch_size


dnn_model = Sequential()
dnn_model.add(Flatten())
dnn_model.add(Dense(128, activation='relu'))
dnn_model.add(Dense(128, activation='relu'))
dnn_model.add(Dense(num_classes, activation='softmax'))

dnn_model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])

print("Done")

(60000, 28, 28, 1)
Done


In [6]:
history = dnn_model.fit(datagen.flow(x_train,y_train,shuffle=True, batch_size = batch),
                                  steps_per_epoch = steps_per_epoch,
                                  epochs=dnn_epochs, batch_size = batch)
print()
print()
predicted_probabilities = dnn_model.predict(x_test)
predicted_classes = np.argmax(predicted_probabilities, axis=1)
acc = 100. * accuracy_score(y_test, predicted_classes)
print("Accuracy: {:.2f}%".format(acc))

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 27ms/step - accuracy: 0.6715 - loss: 1.1501
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step - accuracy: 0.8237 - loss: 0.5981
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 27s 29ms/step - accuracy: 0.8534 - loss: 0.4944
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step - accuracy: 0.8704 - loss: 0.4366
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.8830 - loss: 0.3958
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 33s 22ms/step - accuracy: 0.8931 - loss: 0.3634
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 33s 35ms/step - accuracy: 0.9013 - loss: 0.3372
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 28s 29ms/step - accuracy: 0.9082 - loss: 0.3143
Epoch 9/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step - accuracy: 0.9136 - loss: 0.2948
Epoch 10/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.9195 - loss: 0.2767


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy: 81.34%
